In [1]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("unsloth")
install("trl>=0.15")
install("datasets")
install("bitsandbytes")
install("accelerate")
install("peft")
install("Pillow")
# fmt: on

print("All dependencies installed.")

All dependencies installed.


In [4]:

from dataclasses import dataclass

@dataclass
class Config:
    # Model
    model_name: str = "LiquidAI/LFM2.5-VL-450M"
    max_seq_length: int = 2048
    load_in_4bit: bool = True            # QLoRA — fits on T4 16GB
    
    # LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_target_modules: tuple = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )
    
    # Training
    num_epochs: int = 3
    batch_size: int = 2
    grad_accum_steps: int = 4            # effective batch = 8
    learning_rate: float = 2e-4
    warmup_steps: int = 50
    max_train_samples: int = 2500        # use subset for speed
    
    # Dataset
    dataset_repo: str = "https://github.com/baidongls/MVRSD.git"
    dataset_dir: str = "/content/MVRSD"
    image_size: int = 640                # MVRSD native size
    
    # Output
    output_dir: str = "/content/argus-lfm-lora"
    hub_model_id: str = ""               # set to push to HF Hub

cfg = Config()
print(f"Config ready: {cfg.model_name}, LoRA r={cfg.lora_r}, epochs={cfg.num_epochs}")

Config ready: LiquidAI/LFM2.5-VL-450M, LoRA r=16, epochs=3


In [ ]:

import os, glob, json, random, zipfile, shutil
from pathlib import Path
from PIL import Image

# ── Step 1: Get MVRSD demo data ──────────────────────────────────────────
# The full MVRSD dataset is hosted on Baidu Cloud Drive.
# The GitHub repo contains a demo.zip with sample images + labels.
# We extract that, then supplement with HuggingFace DOTA data.

MVRSD_DIR = cfg.dataset_dir

if not os.path.exists(MVRSD_DIR):
    print("Cloning MVRSD repository...")
    os.system(f"git clone --depth 1 {cfg.dataset_repo} {MVRSD_DIR}")

# Extract demo.zip if it exists
demo_zip = os.path.join(MVRSD_DIR, "demo.zip")
demo_extracted = os.path.join(MVRSD_DIR, "demo")
if os.path.exists(demo_zip) and not os.path.exists(demo_extracted):
    print(f"Extracting demo.zip...")
    with zipfile.ZipFile(demo_zip, "r") as z:
        z.extractall(MVRSD_DIR)
    print("Extracted.")

# ── Step 2: Also pull DOTA from HuggingFace ──────────────────────────────
# HichTala/dota loads natively without custom scripts.
DOTA_DIR = "/content/dota_data"
USE_HF_DOTA = True
hf_dota = None

if USE_HF_DOTA and not os.path.exists(DOTA_DIR):
    print("Downloading DOTA satellite dataset from HuggingFace...")
    try:
        from datasets import load_dataset
        hf_dota = load_dataset("HichTala/dota", split="train")
        os.makedirs(DOTA_DIR, exist_ok=True)
        print(f"  HF DOTA dataset: {len(hf_dota)} samples loaded")
    except Exception as e:
        print(f"  HF DOTA download failed ({e}), using MVRSD only.")
        USE_HF_DOTA = False
        hf_dota = None

# ── Step 3: Class mappings ───────────────────────────────────────────────

MVRSD_CLASSES = {
    0: "Small Military Vehicle",
    1: "Large Military Vehicle",
    2: "Armored Fighting Vehicle",
    3: "Military Construction Vehicle",
    4: "Civilian Vehicle",
}

THREAT_BY_CLASS = {
    0: "MEDIUM",   # SMV
    1: "HIGH",     # LMV
    2: "HIGH",     # AFV
    3: "MEDIUM",   # MCV
    4: "LOW",      # CV
}

REASONING_TEMPLATES = [
    "{label} identified on unpaved road, possible forward deployment",
    "{label} detected near tree cover, likely concealed staging area",
    "{label} observed in open terrain, high visibility exposure",
    "{label} positioned near infrastructure, possible logistics hub",
    "{label} detected in convoy formation, indicates active movement",
    "{label} visible in desert terrain, limited concealment",
    "{label} spotted near airstrip perimeter, possible base security",
    "{label} located in urban area, mixed civilian-military zone",
]

def parse_yolo_annotation(txt_path: str, img_w: int, img_h: int):
    """Parse YOLO-format annotation file -> list of detection dicts."""
    detections = []
    if not os.path.exists(txt_path):
        return detections
    with open(txt_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            cx, cy, w, h = [float(x) for x in parts[1:5]]
            x1 = round(max(0.0, cx - w / 2), 4)
            y1 = round(max(0.0, cy - h / 2), 4)
            x2 = round(min(1.0, cx + w / 2), 4)
            y2 = round(min(1.0, cy + h / 2), 4)

            label = MVRSD_CLASSES.get(cls_id, "Military Vehicle")
            threat = THREAT_BY_CLASS.get(cls_id, "MEDIUM")
            reasoning = random.choice(REASONING_TEMPLATES).format(label=label)

            detections.append({
                "label": label,
                "bbox": [x1, y1, x2, y2],
                "threat_level": threat,
                "confidence": round(random.uniform(0.80, 0.99), 2),
                "reasoning": reasoning,
            })
    return detections


# ── Step 4: Detection prompt (same one used in ARGUS pipeline) ───────────

DETECTION_PROMPT = (
    "You are an orbital intelligence analyst examining satellite imagery "
    "from a defense reconnaissance satellite at ~800 km altitude.\n\n"
    "Detect ALL military-relevant objects visible in this image. "
    "For each object, provide:\n"
    '- "label": specific type of object\n'
    '- "bbox": normalized bounding box [x1, y1, x2, y2] in [0,1]\n'
    '- "threat_level": "LOW", "MEDIUM", or "HIGH"\n'
    '- "confidence": 0.0 to 1.0\n'
    '- "reasoning": brief tactical assessment\n\n'
    'Return a JSON array. If no targets visible, return: []'
)


# ── Step 5: Build conversation pairs ─────────────────────────────────────

def find_image_label_pairs(root_dir: str):
    """Recursively find all (image, label.txt) pairs under root_dir."""
    pairs = []
    all_images = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
        all_images.extend(glob.glob(os.path.join(root_dir, "**", ext), recursive=True))

    for img_path in all_images:
        stem = Path(img_path).stem
        parent = Path(img_path).parent
        # Try sibling labels/ folder
        lbl_folder = parent.parent / "labels"
        txt_path = lbl_folder / f"{stem}.txt"
        if not txt_path.exists():
            # Try same folder
            txt_path = parent / f"{stem}.txt"
        pairs.append((str(img_path), str(txt_path)))

    return pairs


# DOTA class → military label mapping (for HF DOTA dataset)
DOTA_MILITARY_MAP = {
    "plane": ("Military Aircraft", "HIGH"),
    "ship": ("Naval Vessel", "HIGH"),
    "helicopter": ("Helicopter", "HIGH"),
    "large-vehicle": ("Large Military Vehicle", "MEDIUM"),
    "small-vehicle": ("Small Military Vehicle", "LOW"),
    "harbor": ("Harbor Installation", "MEDIUM"),
    "bridge": ("Strategic Bridge", "MEDIUM"),
    "storage-tank": ("Storage Tank", "MEDIUM"),
    "container-crane": ("Port Crane", "LOW"),
    "airport": ("Airfield", "HIGH"),
    "helipad": ("Helipad", "MEDIUM"),
}


def build_dataset_entries():
    """Build VLM training conversation pairs from MVRSD + HF DOTA."""
    entries = []

    # ── Source 1: MVRSD (demo.zip extracted images) ──────────────────
    pairs = find_image_label_pairs(MVRSD_DIR)
    print(f"  Found {len(pairs)} image files in MVRSD directory")

    for img_path, txt_path in pairs:
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            continue

        detections = parse_yolo_annotation(txt_path, img.width, img.height)
        response_json = json.dumps(detections, separators=(",", ":"))

        entries.append({
            "messages": [
                {"role": "user", "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": DETECTION_PROMPT},
                ]},
                {"role": "assistant", "content": [
                    {"type": "text", "text": response_json},
                ]},
            ],
        })

    print(f"  MVRSD entries: {len(entries)}")

    # ── Source 2: HuggingFace DOTA dataset ───────────────────────────
    if hf_dota is not None:
        dota_count = 0
        for row in hf_dota:
            try:
                img = row["image"].convert("RGB") if hasattr(row.get("image", None), "convert") else None
                if img is None:
                    continue

                # Parse DOTA annotations (format varies by HF version)
                objects = row.get("objects", row.get("annotations", {}))
                categories = objects.get("category", objects.get("categories", []))
                bboxes_raw = objects.get("bbox", objects.get("bboxes", []))

                detections = []
                w, h = img.size
                for j, cat in enumerate(categories):
                    label_key = cat if isinstance(cat, str) else str(cat)
                    if label_key not in DOTA_MILITARY_MAP:
                        continue
                    label, threat = DOTA_MILITARY_MAP[label_key]

                    if j < len(bboxes_raw):
                        b = bboxes_raw[j]
                        # Normalize to [0,1] — DOTA bboxes may be pixel coords
                        if max(b) > 1.0:
                            x1, y1 = round(b[0] / w, 4), round(b[1] / h, 4)
                            x2, y2 = round(b[2] / w, 4), round(b[3] / h, 4)
                        else:
                            x1, y1, x2, y2 = b[0], b[1], b[2], b[3]
                    else:
                        continue

                    reasoning = random.choice(REASONING_TEMPLATES).format(label=label)
                    detections.append({
                        "label": label,
                        "bbox": [x1, y1, x2, y2],
                        "threat_level": threat,
                        "confidence": round(random.uniform(0.80, 0.99), 2),
                        "reasoning": reasoning,
                    })

                response_json = json.dumps(detections, separators=(",", ":"))
                entries.append({
                    "messages": [
                        {"role": "user", "content": [
                            {"type": "image", "image": img},
                            {"type": "text", "text": DETECTION_PROMPT},
                        ]},
                        {"role": "assistant", "content": [
                            {"type": "text", "text": response_json},
                        ]},
                    ],
                })
                dota_count += 1
            except Exception:
                continue

        print(f"  DOTA entries: {dota_count}")

    total_with_targets = sum(
        1 for e in entries if '"label"' in e["messages"][1]["content"][0]["text"]
    )
    print(f"  Total: {len(entries)} ({total_with_targets} with targets, {len(entries) - total_with_targets} empty)")

    random.shuffle(entries)
    if cfg.max_train_samples and len(entries) > cfg.max_train_samples:
        entries = entries[:cfg.max_train_samples]
    return entries


print("Building training dataset...")
dataset_entries = build_dataset_entries()
print(f"Dataset ready: {len(dataset_entries)} training samples")

if len(dataset_entries) > 0:
    sample_resp = dataset_entries[0]["messages"][1]["content"][0]["text"]
    print(f"Sample response: {sample_resp[:200]}...")


In [ ]:
%rm -rf /content/MVRSD